# SEC Filings RAG Pipeline: Single-Document Test (PepsiCo 2022 10-K)

End-to-end test of a Retrieval-Augmented Generation (RAG) pipeline on **one** SEC filing before scaling to the full corpus.

## Pipeline overview

| Stage | What happens | Tools |
|-------|--------------|-------|
| 1. Ingestion | Load the enriched, parsed 10-K JSON and build page-level documents | `langchain_core` |
| 2. Chunking | Semantic chunking per page + contextual header for embedding | `SemanticChunker`, `BAAI/bge-small-en-v1.5` |
| 3. Indexing | Store chunk vectors in Qdrant with stable IDs and payload indexes | Qdrant |
| 4. Retrieval | Dense search with **role-based access control** enforced in the vector DB | `QdrantVectorStore` |
| 5. Generation | Answer strictly from retrieved evidence, with citations | Groq `gpt-oss-120b` |
| 6. Debugging | Diagnose missed chunks, query expansion, curated context | Custom helpers |
| 7. Reranking | Rerank dense candidates with Cohere | Cohere Rerank |

**Test question:** *"Has PepsiCo reported any materially important ongoing legal battles from FY2022 and FY2021?"*

## Table of contents
1. [Setup](#1.-Setup)
2. [Data Ingestion](#2.-Data-Ingestion)
3. [Semantic Chunking](#3.-Semantic-Chunking)
4. [Vector Store (Qdrant)](#4.-Vector-Store-(Qdrant))
5. [Retrieval with Role-Based Access](#5.-Retrieval-with-Role-Based-Access)
6. [Answer Generation (LLM)](#6.-Answer-Generation-(LLM))
7. [Retrieval Diagnostics and Improvements](#7.-Retrieval-Diagnostics-and-Improvements)
8. [Reranking with Cohere](#8.-Reranking-with-Cohere)

---
# 1. Setup

Imports, environment variables, and shared configuration.

**Required environment variables** (in `.env`): `QDRANT_URL`, `QDRANT_API_KEY`, `GROQ_API_KEY`, `COHERE_API_KEY`, and optionally `COHERE_RERANK_MODEL`.

In [ ]:
import hashlib
import json
import os
import uuid
from pathlib import Path

import cohere
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_experimental.text_splitter import SemanticChunker
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

load_dotenv()

# Configuration
INPUT_FILE = Path(r"data\enriched\PEPSICO_2022_10K.json")
COLLECTION_NAME = "sec_filings"

---
# 2. Data Ingestion

## 2.1 Load the parsed filing

The enriched JSON contains the document ID, company metadata (sector, desk, doc type, period), and the parsed text for every PDF page.

In [ ]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

doc_id = data["doc_id"]
metadata = data["metadata"]
pages = data["parsed_content"]["pages"]

## 2.2 Build page-level documents

Each non-empty page becomes a LangChain `Document` carrying the metadata needed later for filtering and citations (`doc_id`, `company`, `sector`, `desk`, `doc_type`, `period`, `page`, `source`).

In [ ]:
page_documents = []

for page_data in pages:
    text = page_data.get("text", "").strip()

    if not text:
        continue

    page_documents.append(
        Document(
            page_content=text,
            metadata={
                "doc_id": doc_id,
                "company": metadata["company"],
                "sector": metadata["sector"],
                "desk": metadata["desk"],
                "doc_type": metadata["doc_type"],
                "period": metadata["period"],
                "page": page_data["page"],
                "source": data["parsed_content"]["source_file"],
            },
        )
    )

---
# 3. Semantic Chunking

## 3.1 Embedding model and semantic splitter

- **Embeddings:** `BAAI/bge-small-en-v1.5` on CPU with normalized vectors (suited to cosine similarity).
- **Splitter:** `SemanticChunker` breaks text where embedding distance between neighboring sentences exceeds the 90th percentile.

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
    buffer_size=1,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1165.63it/s]


## 3.2 Split each page into semantic chunks

Pages are chunked **independently** so every chunk keeps an accurate page number for citations.

In [ ]:
semantic_chunks = []

for page_document in page_documents:
    page_chunks = semantic_splitter.split_documents([page_document])
    semantic_chunks.extend(page_chunks)

## 3.3 Enrich chunks with IDs and a contextual header

For each chunk:
- Keep the raw text in `metadata["original_text"]` (used later for display and LLM context).
- Assign a readable `chunk_id`: `{doc_id}_p{page}_c{index}`.
- Prepend a **contextual header** (company, filing, period, doc ID, page) to `page_content`, so the embedding captures document context.

In [ ]:
for index, chunk in enumerate(semantic_chunks, start=1):
    chunk.metadata["original_text"] = chunk.page_content
    chunk.metadata["chunk_id"] = f"{doc_id}_p{chunk.metadata['page']}_c{index:04d}"
    chunk.metadata["content_type"] = "text"

    contextual_header = (
        f"Company: {chunk.metadata['company']}\n"
        f"Filing: {chunk.metadata['doc_type']}\n"
        f"Reporting period: {chunk.metadata['period']}\n"
        f"Document ID: {chunk.metadata['doc_id']}\n"
        f"PDF page: {chunk.metadata['page']}\n"
    )

    # Header is added to the text that gets embedded
    chunk.page_content = contextual_header + "\nContent:\n" + chunk.page_content

## 3.4 Sanity check

Print summary counts and preview the first three chunks.

In [7]:
print("Document:", doc_id)
print("Pages processed:", len(page_documents))
print("Semantic chunks:", len(semantic_chunks))

for chunk in semantic_chunks[:3]:
    print("\n" + "=" * 70)
    print("Chunk ID:", chunk.metadata["chunk_id"])
    print("Page:", chunk.metadata["page"])
    print("Content:")
    print(chunk.page_content[:1500])

Document: PEPSICO_2022_10K
Pages processed: 503
Semantic chunks: 1189

Chunk ID: PEPSICO_2022_10K_p1_c0001
Page: 1
Content:
Company: PepsiCo
Filing: 10k
Reporting period: 2022
Document ID: PEPSICO_2022_10K
PDF page: 1

Content:
Table of Contents 

##### **UNITED STATES SECURITIES AND EXCHANGE COMMISSION** 

**WASHINGTON, D.C. 20549** 

## **FORM 10-K** 

(Mark One) 

☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 

For the fiscal year ended December 31, 2022 

OR 

☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 

For the transition period from to 

Commission file number 1-1183 

PEPSICO Lay's peppsi 

**PepsiCo, Inc.** 

(Exact Name of Registrant as Specified in its Charter) 

**North Carolina** 

(State or Other Jurisdiction of Incorporation or Organization) 

**13-1584302** 

(I.R.S. Employer Identification No.) 

**700 Anderson Hill Road, Purchase, New York 10577** 

(Address of principal executive 

---
# 4. Vector Store (Qdrant)

## 4.1 Connect to Qdrant and create the collection

Creates the `sec_filings` collection (cosine distance) if it does not exist yet. The vector size is inferred from the embedding model.

In [8]:
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)

if not client.collection_exists(COLLECTION_NAME):
    vector_size = len(embeddings.embed_query("Test embedding"))

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE,
        ),
    )

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

print("Connected to remote Qdrant successfully!")

Connected to remote Qdrant successfully!


## 4.2 Generate stable point IDs

IDs are a deterministic UUIDv5 of `doc_id`, page, and a SHA-256 hash of the chunk text. Re-running the notebook therefore **overwrites** the same points instead of creating duplicates.

In [ ]:
point_ids = []

for chunk in semantic_chunks:
    content_hash = hashlib.sha256(
        chunk.metadata["original_text"].encode("utf-8")
    ).hexdigest()

    point_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"{doc_id}:{chunk.metadata['page']}:{content_hash}",
        )
    )

    point_ids.append(point_id)

## 4.3 Insert chunks and verify

Upload in batches of 64, then check the total number of vectors in the collection. The total includes chunks from other documents already indexed.

In [10]:
if semantic_chunks:
    vector_store.add_documents(
        documents=semantic_chunks,
        ids=point_ids,
        batch_size=64,
    )

print(f"Successfully inserted {len(semantic_chunks)} chunks!")

collection_info = client.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Total vectors:", collection_info.points_count)

Successfully inserted 1189 chunks!
Collection: sec_filings
Total vectors: 1670


## 4.4 Create payload indexes

Keyword indexes on `metadata.company` and `metadata.doc_id` make filtered search fast. The access-control filter in Section 5 relies on `metadata.company`.

In [11]:
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="metadata.company",
    field_schema=models.PayloadSchemaType.KEYWORD,
)
print("Created index on metadata.company")

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="metadata.doc_id",
    field_schema=models.PayloadSchemaType.KEYWORD,
)
print("Created index on metadata.doc_id")

collection_info = client.get_collection(COLLECTION_NAME)
print(collection_info.payload_schema)

Created index on metadata.company
Created index on metadata.doc_id
{'metadata.doc_id': PayloadIndexInfo(data_type=<PayloadSchemaType.KEYWORD: 'keyword'>, params=None, points=1670), 'metadata.company': PayloadIndexInfo(data_type=<PayloadSchemaType.KEYWORD: 'keyword'>, params=None, points=1670)}


---
# 5. Retrieval with Role-Based Access

## 5.1 Role access configuration

Each analyst desk may only see filings for its own companies. The `lead` role can see all desks.

| Role | Companies |
|------|-----------|
| `consumer` | PepsiCo, Best Buy, MGM Resorts, Ulta Beauty |
| `health_industrials` | Johnson & Johnson, Pfizer, 3M, Boeing |
| `tech_telecom` | AMD, Adobe, Corning, Verizon |
| `lead` | All of the above |

In [ ]:
ROLE_ACCESS = {
    "consumer": [
        "PepsiCo",
        "Best Buy",
        "MGM Resorts",
        "Ulta Beauty",
    ],
    "health_industrials": [
        "Johnson & Johnson",
        "Pfizer",
        "3M",
        "Boeing",
    ],
    "tech_telecom": [
        "AMD",
        "Adobe",
        "Corning",
        "Verizon",
    ],
    "lead": [
        "PepsiCo", "Best Buy", "MGM Resorts", "Ulta Beauty",
        "Johnson & Johnson", "Pfizer", "3M", "Boeing",
        "AMD", "Adobe", "Corning", "Verizon",
    ],
}

## 5.2 Retrieval function

`search_filings` applies the role's company list as a **Qdrant filter**, so access control is enforced inside the database rather than after retrieval. It returns doc ID, page, chunk ID, similarity score, and the original chunk text.

In [ ]:
def search_filings(query, role, k=5):
    if role not in ROLE_ACCESS:
        raise ValueError(f"Invalid role: {role}")

    allowed_companies = ROLE_ACCESS[role]

    # Enforce access control inside Qdrant
    search_filter = models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.company",
                match=models.MatchAny(any=allowed_companies),
            )
        ]
    )

    # Similarity search
    results = vector_store.similarity_search_with_score(
        query=query,
        k=k,
        filter=search_filter,
    )

    output = []

    for document, score in results:
        output.append({
            "doc_id": document.metadata["doc_id"],
            "page": document.metadata["page"],
            "chunk_id": document.metadata["chunk_id"],
            "score": float(score),
            "text": document.metadata.get("original_text", document.page_content),
        })

    return output

## 5.3 Test: access control

The `health_industrials` desk asks about PepsiCo. Because PepsiCo is not in that role's allowed list, results must come only from permitted companies (e.g. 3M), never PepsiCo.

In [14]:
query = "Has Pepsico reported any materially important ongoing legal battles from FY2022 and FY2021?"

results = search_filings(
    query=query,
    role="health_industrials",
    k=5,
)

for result in results:
    print(json.dumps(result, indent=4))

{
    "doc_id": "3M_2015_10K",
    "page": 12,
    "chunk_id": "3M_2015_10K_p12_c0036",
    "score": 0.6291201,
    "text": "Legal Proceeding** **s.** \n\nDiscussion of legal matters is incorporated by reference from Part II, Item 8, Note 14, \u201cCommitments and Contingencies,\u201d of this document, and should be considered an integral part of Part I, Item 3, \u201cLegal Proceedings.\u201d \n\n## **Item 4. Mine Safety** **Disclosures.** \n\nPursuant to Section 1503 of the Dodd-Frank Wall Street Reform and Consumer Protection Act (the \u201cAct\u201d), the Company is required to disclose, in connection with the mines it operates, information concerning mine safety violations or other regulatory matters in its periodic reports filed with the SEC. For the year 2015, the information concerning mine safety violations or other regulatory matters required by Section 1503(a) of the Act is included in Exhibit 95 to this annual report. 12"
}
{
    "doc_id": "3M_2015_10K",
    "page": 109,
   

---
# 6. Answer Generation (LLM)

## 6.1 Initialize the LLM

Groq-hosted `openai/gpt-oss-120b`.

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
)

## 6.2 System prompt

A strict, evidence-only analyst prompt:
- Use **only** the retrieved filing text and never outside knowledge.
- Analyze each reporting period separately.
- Cite every claim as `[DOCUMENT_ID, PDF page X]`.
- Return `STATUS: REFUSED` when evidence is insufficient, and never treat missing retrieval as evidence of absence.

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are an expert financial research analyst specializing in
SEC filings, including 10-K, 10-Q, 8-K, and earnings reports.

Your task is to answer questions using ONLY the retrieved
evidence from the provided SEC filings.

## CORE RULES

1. Evidence-based answers
- Use only the supplied filing evidence.
- Never use external knowledge or assumptions.
- Never invent facts, figures, events, or citations.
- If evidence is insufficient, explicitly state that.

2. Accurate interpretation
- Interpret disclosures in their original financial and legal context.
- Distinguish between an event existing and its material financial impact.
- Do not interpret "no material adverse effect" as "no litigation."
- Distinguish risks, ongoing events, historical events, and completed events.
- Do not treat a risk-factor disclosure as proof that an event occurred.

3. Multiple reporting periods
- Identify every reporting period requested.
- Analyze each period separately.
- Do not assume that information disclosed in one year also applies
  to another year.
- If evidence for a requested period is missing, explicitly identify it.
- Do not claim that something did not happen merely because retrieval
  returned no evidence.

4. Evidence and citations
- Every factual claim must have a supporting citation.
- Use the exact document ID and PDF page from the evidence.
- Format citations as [DOCUMENT_ID, PDF page X].
- Never fabricate page numbers or document IDs.
- Prefer evidence from the relevant SEC filing section.
- When evidence conflicts, explain the conflict rather than hiding it.

5. Financial data and calculations
- Preserve original units, currencies, and reporting periods.
- Show calculation inputs and formulas when performing calculations.
- Never combine values from different reporting periods without
  explicitly explaining the comparison.

6. Handling insufficient evidence
- If evidence does not support a reliable answer, return REFUSED.
- Clearly explain what evidence is missing.
- Do not fill gaps with assumptions.

## RESPONSE FORMAT

Return a concise, well-structured answer.

For supported questions:
STATUS: ANSWERED

ANSWER:
[Direct answer, followed by relevant explanation.]

EVIDENCE:
- [Relevant finding] [DOCUMENT_ID, PDF page X]

For unsupported questions:
STATUS: REFUSED

ANSWER:
The available filing evidence is insufficient to answer this question.

REASON:
[Explain what evidence is missing.]

## USER QUESTION
{question}

## RETRIEVED SEC FILING EVIDENCE
{context}

## FINAL INSTRUCTION
Analyze the evidence carefully before answering.
Answer the exact question asked, not a related question.
Do not confuse absence of retrieved evidence with evidence of absence.
""")

## 6.3 RAG helpers: `build_context` and `ask_filings`

- `build_context` formats retrieved chunks as numbered evidence blocks with doc ID, page, chunk ID, and score.
- `ask_filings` runs the full flow: **retrieve → build context → format prompt → call LLM → return answer + citations**.

In [ ]:
def build_context(results):
    context_parts = []

    for i, result in enumerate(results, start=1):
        context_parts.append(
            f"""
Evidence {i}
Document ID: {result["doc_id"]}
PDF Page: {result["page"]}
Chunk ID: {result["chunk_id"]}
Similarity Score: {result["score"]}

Text:
{result["text"]}
"""
        )

    return "\n\n".join(context_parts)


def ask_filings(query, role, k=5):
    # 1. Retrieve
    results = search_filings(query=query, role=role, k=k)

    # 2. Build context
    context = build_context(results)

    # 3. Create prompt
    messages = prompt.format_messages(question=query, context=context)

    # 4. Call LLM
    response = llm.invoke(messages)

    # 5. Return answer with citations
    citations = [
        {
            "doc_id": result["doc_id"],
            "page": result["page"],
            "chunk_id": result["chunk_id"],
        }
        for result in results
    ]

    return {
        "status": "answered",
        "answer": response.content,
        "citations": citations,
    }

## 6.4 Test: baseline RAG (k=5 vs k=20)

Ask the PepsiCo legal-proceedings question as the `consumer` role with two retrieval depths.

**Observation:** the answer is incomplete. The key *Legal Proceedings* page (PDF page 26) is not retrieved, so the model refuses. This motivates the diagnostics in Section 7.

In [18]:
query = "Has Pepsico reported any materially important ongoing legal battles from FY2022 and FY2021?"

result = ask_filings(query=query, role="consumer", k=5)
print(json.dumps(result, indent=2))

result = ask_filings(query=query, role="consumer", k=20)
print(result["answer"])

{
  "status": "answered",
  "answer": "**STATUS:** REFUSED  \n\n**ANSWER:**  \nThe available filing evidence does not contain any disclosure of materially important ongoing legal battles for PepsiCo in fiscal year 2022 or fiscal year 2021.\n\n**REASON:**  \nThe retrieved excerpts from PepsiCo\u2019s 2022 Form\u202f10\u2011K include only general statements about applicable laws, intellectual\u2011property considerations, and sign\u2011off signatures. None of the provided pages contain a \u201cLegal Proceedings\u201d or similar section that would identify specific, material, ongoing lawsuits or legal disputes for FY\u202f2022 or FY\u202f2021. Consequently, there is insufficient evidence to determine whether PepsiCo reported any such legal battles in those periods.",
  "citations": [
    {
      "doc_id": "PEPSICO_2022_10K",
      "page": 489,
      "chunk_id": "PEPSICO_2022_10K_p489_c1147"
    },
    {
      "doc_id": "PEPSICO_2022_10K",
      "page": 10,
      "chunk_id": "PEPSICO_2022_

---
# 7. Retrieval Diagnostics and Improvements

## 7.1 Check whether the key page is retrieved

Test whether the *Legal Proceedings* page (PDF page 26) appears in the top 20 dense results.

In [19]:
query = (
    "Has Pepsico reported any materially important ongoing "
    "legal battles from FY2022 and FY2021?"
)

results = search_filings(query=query, role="consumer", k=20)

pages = [result["page"] for result in results]

print("Retrieved pages:", pages)
print("Page 26 retrieved:", 26 in pages)

Retrieved pages: [489, 10, 24, 485, 205, 11, 419, 239, 35, 18, 486, 115, 226, 25, 482, 196, 210, 194, 29, 62]
Page 26 retrieved: False


## 7.2 Query expansion

The plain question is semantically far from the formal wording in 10-K legal sections. `expand_sec_query` appends domain keywords (litigation, legal proceedings, investigations, material adverse effect, etc.) to pull in the right chunks.

In [20]:
def expand_sec_query(question):
    return f"""
    {question}

    Search for information about:
    litigation, legal proceedings, legal claims,
    regulatory proceedings, lawsuits, investigations,
    inquiries, potential liabilities, material litigation,
    material adverse effect, ongoing legal matters.
    """


query = (
    "Has PepsiCo reported any materially important ongoing "
    "legal battles from FY2022 and FY2021?"
)

expanded_query = expand_sec_query(query)

results = search_filings(query=expanded_query, role="consumer", k=10)

for rank, result in enumerate(results, 1):
    print(
        rank,
        result["doc_id"],
        result["page"],
        result["chunk_id"],
        round(result["score"], 4),
    )

1 PEPSICO_2022_10K 25 PEPSICO_2022_10K_p25_c0069 0.7965
2 PEPSICO_2022_10K 24 PEPSICO_2022_10K_p24_c0068 0.789
3 PEPSICO_2022_10K 489 PEPSICO_2022_10K_p489_c1147 0.7824
4 PEPSICO_2022_10K 26 PEPSICO_2022_10K_p26_c0073 0.7806
5 PEPSICO_2022_10K 10 PEPSICO_2022_10K_p10_c0027 0.7704
6 PEPSICO_2022_10K 17 PEPSICO_2022_10K_p17_c0047 0.7668
7 PEPSICO_2022_10K 15 PEPSICO_2022_10K_p15_c0042 0.7665
8 PEPSICO_2022_10K 11 PEPSICO_2022_10K_p11_c0028 0.7659
9 PEPSICO_2022_10K 34 PEPSICO_2022_10K_p34_c0099 0.7634
10 PEPSICO_2022_10K 324 PEPSICO_2022_10K_p324_c0762 0.7617


## 7.3 Answer with curated context and stricter interpretation rules

With the correct chunks (pages 25 and 26) selected manually, a second prompt adds rules about how to interpret legal-proceedings disclosures. The key rule: *"no material adverse effect on financials"* does **not** mean *"no material litigation exists"*.

This tests the **upper bound** of answer quality when retrieval is correct.

In [ ]:
def ask_filings_with_context(question, evidence):
    if not evidence:
        return {
            "status": "refused",
            "answer": "The retrieved evidence is insufficient to answer this question.",
            "citations": [],
        }

    context_parts = []

    for item in evidence:
        context_parts.append(
            f"""
Document: {item['doc_id']}
PDF Page: {item['page']}
Chunk ID: {item['chunk_id']}

Evidence:
{item['text']}
"""
        )

    context = "\n\n".join(context_parts)

    prompt = ChatPromptTemplate.from_template("""
You are a financial research assistant specializing in SEC filings.

Answer the question using ONLY the supplied evidence.
CRITICAL INTERPRETATION RULES:

1. Never conclude that no material legal proceedings exist
   merely because management expects no material adverse
   effect on financial condition, results of operations,
   or cash flows.

2. These are different concepts:
   - Existence of legal proceedings.
   - Materiality of an individual proceeding.
   - Expected aggregate financial impact.

3. If the filing only provides management's assessment of
   financial impact, report that assessment exactly.
   Do not infer individual case materiality.

4. Do not use phrases such as:
   "none of the matters are material"
   "no material lawsuits exist"
   "no material legal battles were reported"
   unless the filing explicitly supports those statements.

5. If the evidence does not establish whether individual
   proceedings are material, explicitly state:
   "The filing does not establish whether any individual
   proceeding is independently material."
Rules:
1. Do not use outside knowledge.
2. Distinguish between the existence of legal proceedings and
   whether their financial impact is considered material.
3. A statement that litigation will not have a material adverse
   effect does NOT mean that no litigation exists.
4. Do not assume that evidence for one financial year supports another.
5. If a requested year is not supported by the evidence,
   explicitly state that it cannot be verified.
6. Cite every factual claim using the document ID and PDF page.
7. Never invent citations.
- Never interpret "will not have a material adverse effect"
  as proof that no individual legal proceeding is material.
- Preserve the exact scope of management's materiality statement.
- Do not convert management's expectation about financial impact
  into a categorical statement about the materiality of every case.
Question:
{question}

Evidence:
{context}

Provide:
- A concise answer.
- Citations for the claims.
- Any limitations in the available evidence.
""")

    response = llm.invoke(
        prompt.format_messages(question=question, context=context)
    )

    return {
        "status": "answered",
        "answer": response.content,
        "citations": [
            {
                "doc_id": item["doc_id"],
                "page": item["page"],
                "chunk_id": item["chunk_id"],
            }
            for item in evidence
        ],
    }

In [22]:
relevant_context = [
    r for r in results
    if r["chunk_id"] in [
        "PEPSICO_2022_10K_p25_c0069",
        "PEPSICO_2022_10K_p26_c0073",
    ]
]

answer = ask_filings_with_context(
    question=query,
    evidence=relevant_context,
)

print(answer)

{'status': 'answered', 'answer': '**FY\u202f2022**\n\nPepsiCo states that it and its subsidiaries are parties to “a variety of litigation, claims, legal or regulatory proceedings, inquiries and investigations” (PEPSICO_2022_10K_p25_c0069).  In the “Item\u202f3\u202fLegal Proceedings” section it adds that “management believes that the final outcome of the foregoing will **not have a material adverse effect** on our financial condition, results of operations or cash flows” (PEPSICO_2022_10K_p26_c0073).  \n\nThe filing therefore confirms the existence of ongoing legal matters, but it does **not** identify any individual proceeding as being “materially important.”  As required by the rules, we must note that the filing does not establish whether any single case is independently material.\n\n**FY\u202f2021**\n\nNo evidence was supplied that covers PepsiCo’s FY\u202f2021 legal‑proceedings disclosures. Consequently, we cannot verify whether PepsiCo reported any materially important ongoing le

---
# 8. Reranking with Cohere

Instead of curating chunks by hand, use a **two-stage retrieval** approach:

1. **Dense retrieval:** BGE embeddings + Qdrant return 30 candidates (fast, high recall).
2. **Reranking:** Cohere Rerank scores each candidate against the original question and keeps the top 10 (slower, higher precision).

## 8.1 Cohere client and rerank function

In [ ]:
co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

RERANK_MODEL = os.getenv("COHERE_RERANK_MODEL", "rerank-v5.0-pro")


def rerank_filings(question, candidates, top_n=10):
    if not candidates:
        return []

    documents = [candidate["text"] for candidate in candidates]

    response = co.rerank(
        model=RERANK_MODEL,
        query=question,
        documents=documents,
        top_n=min(top_n, len(documents)),
    )

    reranked = []

    for rank, item in enumerate(response.results, start=1):
        candidate = candidates[item.index]

        reranked.append({
            **candidate,
            "rerank_rank": rank,
            "rerank_score": float(item.relevance_score),
        })

    return reranked

## 8.2 Run dense retrieval, then rerank

In [24]:
question = (
    "Has PepsiCo reported any materially important ongoing "
    "legal battles from FY2022 and FY2021?"
)

expanded_query = expand_sec_query(question)

# Step 1: Retrieve 30 candidates using BGE + Qdrant
dense_results = search_filings(query=expanded_query, role="consumer", k=30)

# Step 2: Rerank candidates using Cohere
reranked_results = rerank_filings(
    question=question,
    candidates=dense_results,
    top_n=10,
)

# Step 3: Display reranked results
for item in reranked_results:
    print(
        item["rerank_rank"],
        item["doc_id"],
        item["page"],
        item["chunk_id"],
        round(item["rerank_score"], 4),
    )

1 PEPSICO_2022_10K 25 PEPSICO_2022_10K_p25_c0069 0.1496
2 PEPSICO_2022_10K 26 PEPSICO_2022_10K_p26_c0073 0.1175
3 PEPSICO_2022_10K 324 PEPSICO_2022_10K_p324_c0762 0.0848
4 PEPSICO_2022_10K 12 PEPSICO_2022_10K_p12_c0031 0.0678
5 PEPSICO_2022_10K 11 PEPSICO_2022_10K_p11_c0028 0.0365
6 PEPSICO_2022_10K 203 PEPSICO_2022_10K_p203_c0514 0.0347
7 PEPSICO_2022_10K 237 PEPSICO_2022_10K_p237_c0590 0.0347
8 PEPSICO_2022_10K 35 PEPSICO_2022_10K_p35_c0102 0.0308
9 PEPSICO_2022_10K 443 PEPSICO_2022_10K_p443_c1015 0.0263
10 PEPSICO_2022_10K 24 PEPSICO_2022_10K_p24_c0068 0.0251


## 8.3 Compare dense retrieval vs reranking

Track where the two key legal-proceedings chunks (pages 25 and 26) rank before and after reranking.

In [25]:
target_chunks = {
    "PEPSICO_2022_10K_p25_c0069",
    "PEPSICO_2022_10K_p26_c0073",
}

print("DENSE RETRIEVAL")
for rank, item in enumerate(dense_results, start=1):
    if item["chunk_id"] in target_chunks:
        print("Rank:", rank, item["chunk_id"])

print("\nCOHERE RERANKING")
for item in reranked_results:
    if item["chunk_id"] in target_chunks:
        print(
            "Rank:", item["rerank_rank"],
            item["chunk_id"],
            "Score:", round(item["rerank_score"], 4),
        )

DENSE RETRIEVAL
Rank: 1 PEPSICO_2022_10K_p25_c0069
Rank: 4 PEPSICO_2022_10K_p26_c0073

COHERE RERANKING
Rank: 1 PEPSICO_2022_10K_p25_c0069 Score: 0.1496
Rank: 2 PEPSICO_2022_10K_p26_c0073 Score: 0.1175
